# Conway's Game of Life Analysis

Starter notebook for Phase 6. It expects CSV exports from the dashboard to live in `analysis/data/`.


In [ ]:
from pathlib import Path
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

sns.set_theme(style='darkgrid')
warnings.filterwarnings('ignore')


In [ ]:
data_dir = Path('data')
files = sorted(glob.glob(str(data_dir / '*.csv')))

if not files:
    raise FileNotFoundError('No CSV files found in analysis/data/. Export from the dashboard first.')

df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

required_columns = {
    'run_id', 'session', 'density', 'peakPop', 'avgEntropy',
    'avgPBirth', 'avgPDeath', 'generations', 'endReason', 'autocorr'
}

missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing expected columns: {sorted(missing_columns)}')

reason_map = {'stagnant': 0, 'died': 1, 'max_gens': 2}
df['endReasonCode'] = df['endReason'].map(reason_map)

numeric_columns = ['session', 'density', 'peakPop', 'avgEntropy', 'avgPBirth', 'avgPDeath', 'generations', 'autocorr']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df = df.dropna(subset=['density', 'peakPop', 'avgEntropy', 'avgPBirth', 'avgPDeath', 'generations', 'autocorr'])
df['endReasonCode'] = df['endReasonCode'].fillna(-1).astype(int)

print(f'Loaded {len(files)} file(s) and {len(df)} sessions')
display(df.head())
display(df.describe(include='all'))
print('Sessions per density:')
display(df['density'].value_counts().sort_index())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=df, x='density', y='avgEntropy', hue='endReason', ax=axes[0, 0])
axes[0, 0].set(title='Entropy vs Density', xlabel='Density %', ylabel='Average Entropy')

sns.scatterplot(data=df, x='density', y='autocorr', hue='endReason', ax=axes[0, 1])
axes[0, 1].set(title='Autocorrelation vs Density', xlabel='Density %', ylabel='Autocorrelation')

sns.scatterplot(data=df, x='generations', y='avgEntropy', hue='endReason', ax=axes[1, 0])
axes[1, 0].set(title='Entropy vs Generations', xlabel='Generations', ylabel='Average Entropy')

sns.scatterplot(data=df, x='density', y='peakPop', hue='endReason', ax=axes[1, 1])
axes[1, 1].set(title='Peak Population vs Density', xlabel='Density %', ylabel='Peak Population')

plt.tight_layout()
plt.show()


In [ ]:
print('Statistical checks')

if len(df) >= 3:
    r, p = stats.pearsonr(df['density'], df['autocorr'])
    print(f'Pearson density/autocorr: r={r:.3f}, p={p:.4f}')
else:
    print('Not enough rows for Pearson correlation.')

mid = df[(df['density'] >= 30) & (df['density'] <= 40)]['avgPBirth'].dropna()
if len(mid) >= 3:
    stat, p = stats.shapiro(mid)
    print(f'Shapiro mid-density avgPBirth: W={stat:.3f}, p={p:.4f}')
else:
    print('Not enough mid-density rows for Shapiro-Wilk.')

low = df[df['density'] <= 25]['avgPDeath'].dropna()
high = df[df['density'] >= 45]['avgPDeath'].dropna()
if len(low) >= 2 and len(high) >= 2:
    stat, p = stats.ks_2samp(low, high)
    print(f'KS low vs high avgPDeath: statistic={stat:.3f}, p={p:.4f}')
else:
    print('Not enough low/high density rows for KS test.')


In [ ]:
feature_columns = ['density', 'avgEntropy', 'avgPBirth', 'avgPDeath', 'autocorr']
X = df[feature_columns]
y = df['endReason']

class_counts = y.value_counts()
min_class_count = int(class_counts.min()) if not class_counts.empty else 0

if len(class_counts) >= 2 and min_class_count >= 2:
    n_splits = min(5, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    print(f'Classification with {n_splits}-fold CV')

    for name, model in [
        ('Decision Tree', DecisionTreeClassifier(max_depth=3, random_state=42)),
        ('Logistic Regression', LogisticRegression(max_iter=1000)),
        ('k-NN', KNeighborsClassifier(n_neighbors=min(5, max(1, len(df) - 1))))
    ]:
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
        print(f'{name}: {scores.mean():.3f} ± {scores.std():.3f}')
else:
    print('Not enough balanced end-reason classes for classification yet.')

if len(df) >= 5:
    reg_X = df[['density']]
    reg_y = df['generations']
    reg_splits = min(5, len(df))
    reg_cv = KFold(n_splits=reg_splits, shuffle=True, random_state=42)

    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=3)),
        ('reg', LinearRegression()),
    ])

    scores = cross_val_score(poly_model, reg_X, reg_y, cv=reg_cv, scoring='r2')
    print(f'Polynomial lifespan model R²: {scores.mean():.3f} ± {scores.std():.3f}')

    poly_model.fit(reg_X, reg_y)
    density_range = pd.DataFrame({'density': np.arange(15, 56)})
    y_pred = poly_model.predict(density_range)

    plt.figure(figsize=(8, 5))
    plt.scatter(df['density'], df['generations'], alpha=0.55, label='Observed')
    plt.plot(density_range['density'], y_pred, color='crimson', label='Polynomial fit')
    plt.xlabel('Density %')
    plt.ylabel('Generations')
    plt.title('Session Lifespan vs Density')
    plt.legend()
    plt.show()
else:
    print('Not enough rows for regression yet.')


## Next steps

- Add more CSV exports into `analysis/data/` as you complete more sweep runs.
- Once you have enough rows, save trained models into `analysis/models/` with `joblib`.
- If class balance is poor, collect more sessions at underrepresented densities before trusting the classifiers.
